**Cell #01**

# RAG11 Nutrition — Stage 1.9: Verify All Data

Standalone integrity check between the local `rag11_data_sources` /
chunk files and what's actually sitting in
(`stage1_eda_output/source{1,2,3}/*.json`) and what's actually sitting in
`rag11_data_sources` / `rag11_chunks_parent_table` / `rag11_chunks_child_table`.
Previously this
was a small "row counts + one smoke-test query" cell at the end of
`stage1_2_eda_load_chunks.ipynb`; it's pulled out here and expanded into a
real per-record comparison, run independently whenever you want to confirm
a load actually landed correctly (including after a partial/checkpointed
run, or after re-tuning stage1_1 and reloading).

What it checks, for every single chunk file, not just row counts:
- the row exists in Supabase at all (nothing silently missing)
- its `rowJSON` matches the local file's content exactly (nothing corrupted
  or stale from a previous run with different section boundaries)
- its `rowOwnerGUID` / `orderInList` match what the file implies
- (child rows only) an embedding is actually present and the right length
- nothing extra is sitting in the tables that no longer has a local file
  behind it (orphaned rows from before a section-count change)

Read-only throughout — this notebook never writes to Supabase.


In [1]:
# Cell #02
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


**Cell #03**

## Imports & client setup

In [2]:
# Cell #04
import os
import re
import json
import uuid
import resource
from pathlib import Path

from dotenv import load_dotenv
from supabase import create_client, Client

load_dotenv()


def require_env(name: str) -> str:
    value = os.environ.get(name, "").strip()
    if not value:
        raise RuntimeError(
            f"{name} is empty in your .env file. Open .env in the RAG11 folder "
            f"and paste your actual value in after '{name}='."
        )
    return value


SUPABASE_URL = require_env("PUBLIC_SUPABASE_URL")
SUPABASE_KEY = os.environ.get("SUPABASE_SERVICE_ROLE_KEY", "").strip() or require_env("PUBLIC_SUPABASE_ANON_KEY")
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

OUTPUT_ROOT = Path(".") / "stage1_eda_output"
SOURCES_MANIFEST_DIR = OUTPUT_ROOT / "sources"
# Discovered from disk instead of hardcoded, matching stage1_2.
SOURCE_KEYS = sorted(
    (p.name for p in OUTPUT_ROOT.iterdir() if p.is_dir() and re.match(r"^source\d+$", p.name)),
    key=lambda k: int(k[len("source"):]),
)
SOURCES_TABLE = "rag11_data_sources"
PARENT_TABLE = "rag11_chunks_parent_table"
CHILD_TABLE = "rag11_chunks_child_table"
EMBEDDING_DIM = 1024   # must match create_sql_tables.sql / stage1_2's EMBEDDING_MODEL

soft_fd_limit, _ = resource.getrlimit(resource.RLIMIT_NOFILE)
print(f"This kernel process's open-file limit: soft={soft_fd_limit}")
print("Clients ready. Supabase project:", SUPABASE_URL)


This kernel process's open-file limit: soft=1048576
Clients ready. Supabase project: https://czgrxgzdmodkkmbmraub.supabase.co


**Cell #05**

## Load local chunk files + recompute their expected row shape

Same file-parsing and deterministic-`uuid5` logic as
`stage1_2_eda_load_chunks.ipynb`, duplicated here so this notebook is
self-contained and can be run independently, any time, without re-running
the loader.

In [3]:
# Cell #06
PARENT_FILE_RE = re.compile(r"^parent_chunk-(\d+)\.json$")
CHILD_FILE_RE = re.compile(r"^child_chunk-parent(\d+)-chunk(\d+)\.json$")


def load_parent_files(source_key: str) -> list[tuple[int, dict]]:
    out = []
    for f in (OUTPUT_ROOT / source_key).glob("parent_chunk-*.json"):
        m = PARENT_FILE_RE.match(f.name)
        if not m:
            continue
        out.append((int(m.group(1)), json.loads(f.read_text(encoding="utf-8"))))
    out.sort(key=lambda t: t[0])
    return out


def load_child_files(source_key: str) -> list[tuple[int, int, dict]]:
    out = []
    for f in (OUTPUT_ROOT / source_key).glob("child_chunk-*.json"):
        m = CHILD_FILE_RE.match(f.name)
        if not m:
            continue
        out.append((int(m.group(1)), int(m.group(2)), json.loads(f.read_text(encoding="utf-8"))))
    out.sort(key=lambda t: (t[0], t[1]))
    return out


parents_by_source = {k: load_parent_files(k) for k in SOURCE_KEYS}
children_by_source = {k: load_child_files(k) for k in SOURCE_KEYS}
for k in SOURCE_KEYS:
    print(f"[{k}] {len(parents_by_source[k])} local parent file(s), "
          f"{len(children_by_source[k])} local child file(s)")

RAG11_UUID_NAMESPACE = uuid.uuid5(uuid.NAMESPACE_DNS, "rag11.nutrition.poc")


def deterministic_uuid(business_key: str) -> str:
    return str(uuid.uuid5(RAG11_UUID_NAMESPACE, business_key))


# source_key -> its rag11_data_sources rowGUID (== the rowOwnerGUID expected
# on every parent/child row for that source). Read from the manifest
# stage1_1_extract_and_chunk.ipynb writes, rather than recomputed here, so
# this notebook never has to know stage1_1's file_id-based uuid5 seed.
source_rows_local = [
    json.loads(f.read_text(encoding="utf-8"))
    for f in sorted(SOURCES_MANIFEST_DIR.glob("source_row-*.json"))
]
source_row_guid_by_key = {r["rowJSON"]["source_key"]: r["rowGUID"] for r in source_rows_local}

missing_manifest = [k for k in SOURCE_KEYS if k not in source_row_guid_by_key]
if missing_manifest:
    raise RuntimeError(
        f"No rag11_data_sources manifest found for {missing_manifest} in "
        f"{SOURCES_MANIFEST_DIR} -- re-run stage1_1_extract_and_chunk.ipynb."
    )

# expected_parent[rowGUID] = (source_key, rowOwnerGUID, orderInList, rowJSON-from-file)
expected_parent = {}
for source_key in SOURCE_KEYS:
    owner_guid = source_row_guid_by_key[source_key]
    for order, data in parents_by_source[source_key]:
        guid = deterministic_uuid(f"parent:{data['parent_id']}")
        expected_parent[guid] = (source_key, owner_guid, order, data)

# expected_child[rowGUID] = (source_key, rowOwnerGUID, rowParentGUID, orderInList, rowJSON-from-file)
expected_child = {}
for source_key in SOURCE_KEYS:
    owner_guid = source_row_guid_by_key[source_key]
    for _p_order, c_order, data in children_by_source[source_key]:
        guid = deterministic_uuid(f"child:{data['child_id']}")
        parent_guid = deterministic_uuid(f"parent:{data['parent_id']}")
        expected_child[guid] = (source_key, owner_guid, parent_guid, c_order, data)

print(f"\nExpected from local files: {len(source_rows_local)} source row(s), "
      f"{len(expected_parent)} parent row(s), {len(expected_child)} child row(s)")


[source1] 133 local parent file(s), 8 local child file(s)
[source2] 136 local parent file(s), 184 local child file(s)
[source3] 91 local parent file(s), 185 local child file(s)
[source4] 15 local parent file(s), 46 local child file(s)
[source5] 22 local parent file(s), 106 local child file(s)
[source6] 22 local parent file(s), 106 local child file(s)
[source7] 94 local parent file(s), 75 local child file(s)
[source8] 97 local parent file(s), 294 local child file(s)
[source9] 35 local parent file(s), 173 local child file(s)
[source10] 9 local parent file(s), 101 local child file(s)
[source11] 30 local parent file(s), 712 local child file(s)
[source12] 15 local parent file(s), 125 local child file(s)
[source13] 28 local parent file(s), 195 local child file(s)
[source14] 18 local parent file(s), 123 local child file(s)
[source15] 42 local parent file(s), 188 local child file(s)
[source16] 245 local parent file(s), 300 local child file(s)
[source17] 15 local parent file(s), 192 local child

**Cell #07**

## Fetch every row from Supabase (paginated)

`select()` is capped per request, so this pages through with `.range()`
until a page comes back short — works the same whether the table has a
few hundred rows or a few hundred thousand.

In [4]:
# Cell #08
def fetch_all_rows(table_name: str, columns: str, page_size: int = 1000) -> dict:
    """Return {rowGUID: row_dict} for every row in `table_name`."""
    rows_by_guid = {}
    start = 0
    while True:
        resp = (
            supabase.table(table_name)
            .select(columns)
            .range(start, start + page_size - 1)
            .execute()
        )
        page = resp.data
        for row in page:
            rows_by_guid[row["rowGUID"]] = row
        if len(page) < page_size:
            break
        start += page_size
    return rows_by_guid


print("Fetching all source rows from Supabase...")
db_sources = fetch_all_rows(
    SOURCES_TABLE, '"rowGUID","rowOwnerGUID","rowParentGUID","orderInList","rowJSON"'
)
print(f"  -> {len(db_sources)} row(s) in {SOURCES_TABLE}")

print("Fetching all parent rows from Supabase...")
db_parent = fetch_all_rows(PARENT_TABLE, '"rowGUID","rowOwnerGUID","orderInList","rowJSON"')
print(f"  -> {len(db_parent)} row(s) in {PARENT_TABLE}")

print("Fetching all child rows from Supabase (including embeddings)...")
db_child = fetch_all_rows(
    CHILD_TABLE,
    '"rowGUID","rowOwnerGUID","rowParentGUID","orderInList","rowJSON","embedding"',
)
print(f"  -> {len(db_child)} row(s) in {CHILD_TABLE}")


Fetching all source rows from Supabase...
  -> 17 row(s) in rag11_data_sources
Fetching all parent rows from Supabase...
  -> 1047 row(s) in rag11_chunks_parent_table
Fetching all child rows from Supabase (including embeddings)...
  -> 3113 row(s) in rag11_chunks_child_table


**Cell #09**

## Source rows -- local manifest vs `rag11_data_sources`


In [5]:
# Cell #10
source_missing = []
source_mismatched = []
source_ok = 0

for local_row in source_rows_local:
    guid = local_row["rowGUID"]
    row = db_sources.get(guid)
    source_key = local_row["rowJSON"]["source_key"]
    if row is None:
        source_missing.append(source_key)
        continue
    if (row["rowJSON"] != local_row["rowJSON"]
            or row["rowOwnerGUID"] != local_row["rowOwnerGUID"]
            or row["rowParentGUID"] is not None
            or row["orderInList"] != local_row["orderInList"]):
        source_mismatched.append(source_key)
        continue
    source_ok += 1

local_source_guids = {r["rowGUID"] for r in source_rows_local}
source_orphaned = [guid for guid in db_sources if guid not in local_source_guids]

print(f"Source rows -- OK: {source_ok}, missing: {len(source_missing)}, "
      f"mismatched: {len(source_mismatched)}, orphaned in DB: {len(source_orphaned)}")
if source_missing:
    print(f"  missing: {source_missing}")
if source_mismatched:
    print(f"  mismatched: {source_mismatched}")


Source rows -- OK: 17, missing: 0, mismatched: 0, orphaned in DB: 0


**Cell #11**

## Compare — parent rows

In [6]:
# Cell #12
def embedding_length(value) -> int:
    """pgvector comes back over PostgREST as either a JSON list of floats or
    a "[0.1,0.2,...]" string depending on client/library versions -- handle
    both so this check doesn't depend on which one you have installed."""
    if value is None:
        return 0
    if isinstance(value, list):
        return len(value)
    if isinstance(value, str):
        return value.count(",") + 1 if value.strip("[]") else 0
    return 0


parent_missing = []      # local file exists, no row in Supabase
parent_mismatched = []   # row exists, but content differs from the local file
parent_ok = 0

for guid, (source_key, owner, order, local_json) in expected_parent.items():
    row = db_parent.get(guid)
    if row is None:
        parent_missing.append((source_key, order, local_json.get("parent_id")))
        continue
    if row["rowJSON"] != local_json or row["rowOwnerGUID"] != owner or row["orderInList"] != order:
        parent_mismatched.append((source_key, order, local_json.get("parent_id")))
        continue
    parent_ok += 1

parent_orphaned = [guid for guid in db_parent if guid not in expected_parent]

print(f"Parent rows -- OK: {parent_ok}, missing: {len(parent_missing)}, "
      f"mismatched: {len(parent_mismatched)}, orphaned in DB: {len(parent_orphaned)}")

for label, items in [("missing", parent_missing), ("mismatched", parent_mismatched)]:
    if items:
        print(f"\n  First {min(5, len(items))} {label} parent row(s):")
        for source_key, order, parent_id in items[:5]:
            print(f"    [{source_key} #{order}] {parent_id}")


Parent rows -- OK: 1047, missing: 0, mismatched: 0, orphaned in DB: 0


**Cell #13**

## Compare — child rows (+ embedding presence/dimension)

In [7]:
# Cell #14
child_missing = []
child_mismatched = []
child_missing_embedding = []
child_ok = 0

for guid, (source_key, owner, parent_guid, order, local_json) in expected_child.items():
    row = db_child.get(guid)
    if row is None:
        child_missing.append((source_key, order, local_json.get("child_id")))
        continue
    content_matches = (
        row["rowJSON"] == local_json
        and row["rowOwnerGUID"] == owner
        and row["rowParentGUID"] == parent_guid
        and row["orderInList"] == order
    )
    if not content_matches:
        child_mismatched.append((source_key, order, local_json.get("child_id")))
        continue
    if embedding_length(row.get("embedding")) != EMBEDDING_DIM:
        child_missing_embedding.append((source_key, order, local_json.get("child_id")))
        continue
    child_ok += 1

child_orphaned = [guid for guid in db_child if guid not in expected_child]

print(f"Child rows -- OK: {child_ok}, missing: {len(child_missing)}, "
      f"mismatched: {len(child_mismatched)}, bad/missing embedding: {len(child_missing_embedding)}, "
      f"orphaned in DB: {len(child_orphaned)}")

for label, items in [
    ("missing", child_missing),
    ("mismatched", child_mismatched),
    ("with a bad/missing embedding", child_missing_embedding),
]:
    if items:
        print(f"\n  First {min(5, len(items))} child row(s) {label}:")
        for source_key, order, child_id in items[:5]:
            print(f"    [{source_key} #{order}] {child_id}")


Child rows -- OK: 3113, missing: 0, mismatched: 0, bad/missing embedding: 0, orphaned in DB: 0


**Cell #15**

## Row-count summary by source

Cross-checks local file counts against Supabase counts per
`rowOwnerGUID`, which is usually the fastest way to spot a stale/orphaned
source after re-tuning stage1_1 (e.g. Source 2 shrinking from 426 to 77
sections after the running-header fix).

In [8]:
# Cell #16
import html
from collections import Counter

from IPython.display import display, HTML

local_parent_counts = Counter(source_key for source_key, _, _, _ in expected_parent.values())
db_parent_counts = Counter(row["rowJSON"].get("source_key", row["rowOwnerGUID"]) for row in db_parent.values())
local_child_counts = Counter(source_key for source_key, _, _, _, _ in expected_child.values())
db_child_counts = Counter(row["rowJSON"].get("source_key", row["rowOwnerGUID"]) for row in db_child.values())

# difference_parents/difference_children = local - db. Zero means an exact
# match; nonzero pinpoints which source and which table (parent vs child)
# drifted, and the sign says which side has more rows (positive -> local
# has rows Supabase doesn't, e.g. not yet upserted; negative -> Supabase has
# rows local doesn't, e.g. orphaned from a since-shrunk source).
table_lines = [
    f"{'source':<10} {'local parents':>14} {'db parents':>11} {'difference_parents':>19} "
    f"{'local children':>15} {'db children':>12} {'difference_children':>20}"
]
for source_key in SOURCE_KEYS:
    difference_parents = local_parent_counts[source_key] - db_parent_counts[source_key]
    difference_children = local_child_counts[source_key] - db_child_counts[source_key]
    flag = "" if (difference_parents == 0 and difference_children == 0) else "  <-- mismatch"
    table_lines.append(
        f"{source_key:<10} {local_parent_counts[source_key]:>14} {db_parent_counts[source_key]:>11} "
        f"{difference_parents:>19} {local_child_counts[source_key]:>15} {db_child_counts[source_key]:>12} "
        f"{difference_children:>20}{flag}"
    )

# Rendered as HTML (instead of a plain print()) so the table sits in its own
# horizontally scrollable box -- with 7 columns it's wider than the notebook
# output pane, and this way it scrolls in place instead of wrapping mid-row
# or getting clipped. white-space:pre keeps every line's fixed-width
# alignment intact; overflow-x:auto is what makes the box scroll.
display(HTML(
    '<div style="overflow-x:auto; max-width:100%; border:1px solid #8888; padding:4px 0;">'
    f'<pre style="margin:0; font-family:monospace; white-space:pre; padding:0 8px;">'
    f'{html.escape(chr(10).join(table_lines))}</pre>'
    '</div>'
))


**Cell #17**

## Smoke-test vector search

Confirms `match_rag11_child_chunks` (the RPC from `create_sql_tables.sql`)
actually returns results end to end, using a real embedding already
stored in the table (no Voyage API call needed here).

In [9]:
# Cell #18
sample_child = next(iter(db_child.values()), None)
if sample_child is None:
    print("No child rows in the table yet -- nothing to smoke-test.")
else:
    query_embedding = sample_child["embedding"]
    resp = supabase.rpc("match_rag11_child_chunks", {
        "query_embedding": query_embedding,
        "match_count": 3,
    }).execute()
    print("Smoke-test match_rag11_child_chunks (querying with one child's own embedding):")
    for row in resp.data:
        preview = row["rowJSON"]["text"][:80].replace("\n", " ")
        print(f"  [{row['rowJSON'].get('source_key', row['rowOwnerGUID'])} #{row['orderInList']}] "
              f"dist={row['cosine_distance']:.4f}  {preview}...")


Smoke-test match_rag11_child_chunks (querying with one child's own embedding):
  [source1 #1] dist=0.0000  [Source: human-nutrition-text.pdf | Section: Introduction | Pages 45-45]  Image ...
  [source1 #1] dist=0.2241  [Source: human-nutrition-text.pdf | Section: Introduction | Pages 97-97]  Image ...
  [source3 #1] dist=0.3399  [Source: Nutrition-Science-and-Everyday-Application-1773787282.pdf | Section: Un...


**Cell #19**

## Overall verdict

In [10]:
# Cell #20
total_issues = (
    len(source_missing) + len(source_mismatched) + len(source_orphaned)
    + len(parent_missing) + len(parent_mismatched) + len(parent_orphaned)
    + len(child_missing) + len(child_mismatched) + len(child_missing_embedding) + len(child_orphaned)
)

if total_issues == 0:
    print("PASS -- every local source/chunk file matches its row in Supabase exactly, "
          "and every child row has a valid embedding. No orphaned rows either.")
else:
    print(f"FAIL -- {total_issues} issue(s) found across the checks above.")
    if source_orphaned or parent_orphaned or child_orphaned:
        print(f"  {len(source_orphaned)} orphaned source row(s) + {len(parent_orphaned)} orphaned "
              f"parent row(s) + {len(child_orphaned)} orphaned child row(s) in Supabase have no "
              f"matching local file -- likely leftovers from before a source/section-count change. "
              f"See delete_chunks_data.sql to clear them, then re-run stage1_2_eda_load_chunks.ipynb.")
    if child_missing or parent_missing or source_missing:
        print("  Some local chunks never made it into Supabase -- re-run "
              "stage1_2_eda_load_chunks.ipynb (its checkpointing will skip what already succeeded).")


PASS -- every local source/chunk file matches its row in Supabase exactly, and every child row has a valid embedding. No orphaned rows either.
